In [ ]:
import cv2
import gymnasium as gym
import numpy as np
from tetris_gymnasium.envs.tetris import Tetris
from tetris_gymnasium.wrappers.observation import RgbObservation, FeatureVectorObservation
from tetris_gymnasium.wrappers.grouped import GroupedActionsObservations
from gymnasium.wrappers import TimeLimit, ResizeObservation, RecordVideo, MaxAndSkipObservation
from stable_baselines3 import DQN, PPO
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T
from torch.optim.lr_scheduler import ExponentialLR

In [3]:
RENDER_ENV = False
Render_Frame_rate=4
RESIZE_ENV = False
new_size = (56,80)
batch_size = 32
num_episodes = 43200
max_episode_steps = 100
num_stacked_frames = 4
num_frame_skip = 2

In [5]:
def calc_max_height(mat, height=1):
  mat1 = np.rot90(mat)
  mat1 = np.rot90(mat1)
  act_height=0
  not_seen_height=0
  for row in mat:
    act_height+=1
    if act_height < height-4:
      continue
    if act_height > height+4:
      return(height)
    for col in row:
      if col > 0:
        return(act_height)
  return height

def calc_holes(mat,height):
  mat = np.rot90(mat)
  holes = 0
  for row in mat:
    countdown = 20-height
    flag = False
    for col in row:
      countdown-=1
      if countdown < 0:
        if col > 0:
          flag = True
        elif flag:
          holes+=1
  return holes

In [6]:
try:
  env.close()
except:
  print('no hay env para cerrar')

no hay env para cerrar


In [ ]:
class ConvDQN(nn.Module):
    def __init__(self, input_shape, num_actions):
        super(ConvDQN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4),
            nn.LeakyReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.LeakyReLU(),
            nn.Conv2d(64, 64, kernel_size=2, stride=1),
            nn.LeakyReLU()
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(self.calc_conv_output(input_shape), 512),
            nn.LeakyReLU(),
            nn.Linear(512, num_actions)
        )

    def calc_conv_output(self, shape):
        dummy_input = torch.zeros(1, *shape)
        dummy_output = self.conv_layers(dummy_input)
        return int(np.prod(dummy_output.size()))

    def forward(self, x):
        conv_out = self.conv_layers(x).view(x.size()[0], -1)
        return self.fc_layers(conv_out)

In [ ]:
class ConvDQNAgent:
    def __init__(self, input_shape, num_actions, lr, gamma, epsilon, epsilon_decay, buffer_size):
        self.input_shape = input_shape
        self.num_actions = num_actions
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.memory = deque(maxlen=buffer_size)
        self.device = 'cuda'
        self.model = ConvDQN(input_shape, num_actions).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

    def preprocess(self, state):
        state = torch.tensor(state, dtype=torch.float32, device=self.device)
        transform = T.Lambda(lambda x: x.permute(0,3,1,2).reshape(-1, self.input_shape[1], self.input_shape[2]))
        return transform(state)
    
    def preprocess_wv(self, state):
        state_tensor = torch.tensor(state, dtype=torch.float32, device=self.device)
        state_tensor = state_tensor / 255.0
        state_tensor = state_tensor.permute(0, 3, 1, 2) 
        C_out = self.input_shape[0]
        H_out = self.input_shape[1] 
        W_out = self.input_shape[2] 
        state_tensor = state_tensor.contiguous().view(C_out, H_out, W_out)
        return state_tensor
    
    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return np.random.choice(self.num_actions)
        state = self.preprocess_wv(state)
        with torch.no_grad():
            q_values = self.model(state.unsqueeze(0))
        return torch.argmax(q_values).item()

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                next_state = self.preprocess_wv(next_state)
                target = reward + self.gamma * torch.max(self.model(next_state.unsqueeze(0))).item()
            state = self.preprocess_wv(state)
            target_f = self.model(state.unsqueeze(0)).to("cpu").detach().numpy()
            target_f[0][action] = target
            self.optimizer.zero_grad()
            loss = nn.MSELoss()(torch.tensor(target_f).to(self.device), self.model(state.unsqueeze(0)))
            loss.backward()
            self.optimizer.step()
        if self.epsilon > 0.01:
            self.epsilon *= self.epsilon_decay
    def replay_vect(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*minibatch)
        states_tensor = torch.stack([self.preprocess_wv(s) for s in states])
        next_states_tensor = torch.stack([self.preprocess_wv(ns) for ns in next_states])
        actions_tensor = torch.tensor(actions, dtype=torch.long, device=self.device)
        rewards_tensor = torch.tensor(rewards, dtype=torch.float32, device=self.device)
        dones_tensor = torch.tensor(dones, dtype=torch.bool, device=self.device)
        with torch.no_grad():
            next_q_values = self.model(next_states_tensor)
            max_next_q = torch.max(next_q_values, dim=1)[0]
        target_q_values = rewards_tensor + self.gamma * max_next_q * (~dones_tensor)
        current_q_values = self.model(states_tensor)
        current_q_for_actions = current_q_values.gather(1, actions_tensor.unsqueeze(1)).squeeze()
        self.optimizer.zero_grad()
        loss = nn.MSELoss()(current_q_for_actions, target_q_values)
        loss.backward()
        self.optimizer.step()
        if self.epsilon > 0.05:
            self.epsilon *= self.epsilon_decay

In [11]:

class CustomRewardWrapper(gym.RewardWrapper):
    """
    Custom reward shaping to encourage forward movement.
    This wrapper modifies the reward based on the agent's horizontal position.
    """
    def __init__(self, env, holes_penalty=-0.005, heigh_penalty=-0.2):
        super(CustomRewardWrapper, self).__init__(env)
        self.holes_penalty = holes_penalty
        self.heigh_penalty = heigh_penalty

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)

        game_variables = env.unwrapped.get_state().board[:-4, 4:-4]
        self.previous_max_height = calc_max_height(game_variables, 1)

        return obs, info

    def reward(self, reward):
        #print(f"Reward original: {reward}")
        # Probar mayor penalizacion de agujeros
        custom_reward = reward
        game_variables = env.unwrapped.get_state().board[:-4, 4:-4]

        if game_variables.any():
            current_max_height = calc_max_height(game_variables, self.previous_max_height)
            current_holes = calc_holes(game_variables, self.previous_max_height)
            if current_max_height > self.previous_max_height:
              custom_reward+=self.heigh_penalty
            custom_reward += current_holes*self.holes_penalty
            self.previous_max_height = current_max_height
        return custom_reward

if __name__ == "__main__":
    env = gym.make("tetris_gymnasium/Tetris", render_mode="rgb_array")
    #env = FeatureVectorObservation(env)
    env = RgbObservation(env)
    #env = GroupedActionsObservations(env)
    #env = FrameStackObservation(env, stack_size=num_stacked_frames)
    env = CustomRewardWrapper(env)
    env.reset(seed=42)
    #model = DQN("MlpPolicy", env, verbose=1, buffer_size=10000) # para Mlp usar FeatureVectorObservation para Cnn
    model = PPO("CnnPolicy", env, verbose=1) # para Mlp usar FeatureVectorObservation para Cnn
    model.learn(total_timesteps=num_episodes*max_episode_steps, log_interval=4)
    model.save("../Models_Saves/PPO_tetris_1")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


AssertionError: You should use NatureCNN only with images not with Box(0, 7, (24, 34, 3), uint8)
(you are probably using `CnnPolicy` instead of `MlpPolicy` or `MultiInputPolicy`)
If you are using a custom environment,
please check it using our env checker:
https://stable-baselines3.readthedocs.io/en/master/common/env_checker.html.
If you are using `VecNormalize` or already normalized channel-first images you should pass `normalize_images=False`: 
https://stable-baselines3.readthedocs.io/en/master/guide/custom_env.html

In [ ]:
episode = 0
try:
  env = RecordVideo(
    env,
    video_folder='../Video_Tetris_IA',    # Folder to save videos
    name_prefix=f'eval-Ep_testing_{episode}-Trained_steps_{num_episodes*max_episode_steps}',               # Prefix for video filenames
    episode_trigger=lambda x: True    # Record every episode
  )
except:
  print('error implementando grabacion')

In [ ]:
for episode in range(10):
  state, info = env.reset()
  total_reward = 0
  done = False
  step_count = 0
  while not done:
    step_count+=1
    action, _states = model.predict(state, deterministic=True)
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward
  print(f"Episode: {episode} Reward: {total_reward} Steps: {step_count}")

In [ ]:
# var = len(env.unwrapped.get_state().board)
# count = var
# temp = env.unwrapped.get_state().board[:-4, 4:-4]
# print(temp)
# print(calc_max_height(temp))
# print(calc_holes(temp))
# #print(len(env.unwrapped.get_state().board))

In [12]:
from stable_baselines3.common.env_checker import check_env

check_env(env)

/home/seba/Documentos/AI_Juegos/.venv/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:63: UserWarning: It seems that your observation space  is an image but the upper and lower bounds are not in [0, 255]. Because the CNN policy normalize automatically the observation you may encounter issue if the values are not in that range.
  warnings.warn(
/home/seba/Documentos/AI_Juegos/.venv/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:76: UserWarning: The minimal resolution for an image is 36x36 for the default `CnnPolicy`. You might need to use a custom features extractor cf. https://stable-baselines3.readthedocs.io/en/master/guide/custom_policy.html
  warnings.warn(


AssertionError: The observation returned by the `reset()` method does not match the bounds of the given observation space Box(0, 7, (24, 34, 3), uint8). 
1688 invalid indices: 
Expected: 0 <= obs[0,0,0] <= 7, actual value: 128 
Expected: 0 <= obs[0,0,1] <= 7, actual value: 128 
Expected: 0 <= obs[0,0,2] <= 7, actual value: 128 
Expected: 0 <= obs[0,1,0] <= 7, actual value: 128 
Expected: 0 <= obs[0,1,1] <= 7, actual value: 128 
Expected: 0 <= obs[0,1,2] <= 7, actual value: 128 
Expected: 0 <= obs[0,2,0] <= 7, actual value: 128 
Expected: 0 <= obs[0,2,1] <= 7, actual value: 128 
Expected: 0 <= obs[0,2,2] <= 7, actual value: 128 
Expected: 0 <= obs[0,3,0] <= 7, actual value: 128 
Expected: 0 <= obs[0,3,1] <= 7, actual value: 128 
Expected: 0 <= obs[0,3,2] <= 7, actual value: 128 
Expected: 0 <= obs[0,8,2] <= 7, actual value: 240 
Expected: 0 <= obs[0,14,0] <= 7, actual value: 128 
Expected: 0 <= obs[0,14,1] <= 7, actual value: 128 
Expected: 0 <= obs[0,14,2] <= 7, actual value: 128 
Expe